# Final Notebook - K-Means Image Segmentation

## 1. Define Problem

The assignment requires segmenting a given image into `K` clusters using the K-Means algorithm. This project applies unsupervised pixel clustering to landscape images. Each pixel is represented by color-space features, assigned to the nearest centroid with Euclidean distance, and reconstructed by replacing the original pixel color with the centroid color.

This report is notebook-only and contains **0 code cells**. All code execution, model training, figures, labels, metrics, and review artifacts were generated by the project scripts before this notebook was written.

## 2. Assignment Mapping

| Assignment requirement | Concrete implementation |
|---|---|
| Load the image | `src/image_io.py` loads RGB images with PIL |
| Convert color space | RGB, HSV, and LAB are compared |
| Resize image | `max_side=128` for retraining efficiency |
| Flatten pixels | each image becomes `(height * width, n_features)` |
| Apply K-Means | NumPy implementation in `src/kmeans.py` |
| Assign nearest centroid | vectorized Euclidean distance |
| Update centroids | cluster means are recomputed each iteration |
| Repeat until convergence | controlled by `max_iter` and `tol` |
| Segment image | pixels are replaced by centroid colors |
| Visualize result | side-by-side and K-grid figures are exported |

## 3. Workflow With Evidence

The implemented workflow is:

1. **Raw data collection**: image files under `../data/raw/api` and raw manifest `../data/manifest/raw_landscape_manifest.csv`.
2. **Data audit and balancing**: balanced clean manifest `../data/manifest/clean_landscape_manifest.csv` and outlier/source report `../reports/metrics/data_balance_report.json`.
3. **Preprocessing**: RGB loading, resize, color conversion, optional spatial `(x, y)` features.
4. **Training**: K-Means from scratch across `K=2..10`, color spaces `rgb/hsv/lab`, and `use_xy=false/true`.
5. **Outputs**: segmented figures in `../reports/figures`, label arrays in `../data/labels`, and model artifacts in `../models`.
6. **Comparison and review**: metrics `../reports/metrics/model_comparison.csv`, best model report `../reports/metrics/best_model_by_image.json`, and source review `../reports/metrics/source_review.json`.

## 4. Data Audit

The data audit computes width, height, aspect ratio, mean intensity, standard deviation, file size, source, and query for every clean image. IQR and z-score checks identify images that are far from the dataset distribution.

**Audit summary**

| clean_images | raw_images | real_selected | augmentation_selected | max_augmentation | model_runs |
| --- | --- | --- | --- | --- | --- |
| 20 | 22 | 20 | 0 | 8 | 1080 |

**Feature ranges after balancing**

| Feature | Minimum | Mean | Maximum |
|---|---:|---:|---:|
| width | 720 | 2848.90 | 7580 |
| height | 663 | 1899.55 | 4667 |
| mean intensity | 65.87 | 107.26 | 164.67 |
| std intensity | 44.59 | 57.10 | 77.39 |

## 5. Data Balancing

The selected training data keeps 20 clean landscape images. The balance rule prioritizes real Wikimedia/raw images, filters extreme quality outliers when enough candidates exist, and caps augmentation at 8 images. In this final run, the pipeline selected **20 real images** and **0 augmented images**.

**Source distribution**

| source | count |
| --- | --- |
| wikimedia_commons | 4 |
| wikimedia_commons_category | 2 |
| wikimedia_existing_raw | 14 |

**Query distribution**

| query | count |
| --- | --- |
| Desert landscapes | 1 |
| Landscape photographs | 1 |
| beach landscape | 1 |
| existing raw landscape | 14 |
| forest landscape | 1 |
| lake landscape | 1 |
| national park | 1 |

## 6. Preprocessing

Every selected image is resized to `max_side=128`, then represented in RGB, HSV, or LAB. The experiment also compares two feature modes:

- color-only features: `[c1, c2, c3]`
- color + spatial features: `[c1, c2, c3, x, y]`

No ground-truth masks, semantic labels, or deep learning segmentation models are used.

## 7. Training Grid

The training grid is fixed and reproducible:

| Dimension | Values |
|---|---|
| Images | 20 |
| K values | [2, 3, 4, 5, 6, 7, 8, 9, 10] |
| Color spaces | ['rgb', 'hsv', 'lab'] |
| Spatial modes | [False, True] |
| Expected runs | 20 × 9 × 3 × 2 = 1080 |

Each run saves a segmented image, side-by-side comparison, pixel label array, and `.npz` K-Means artifact.

## 8. Model Comparison

The comparison uses unsupervised internal metrics because the assignment does not provide ground-truth segmentation masks.

| Metric | Direction | Meaning |
|---|---|---|
| silhouette_sample | higher | better cluster separation |
| davies_bouldin_sample | lower | lower within/between cluster ratio |
| calinski_harabasz_sample | higher | stronger between-cluster dispersion |
| inertia_per_pixel | lower | pixels closer to assigned centroids |
| cluster_balance | higher | fewer collapsed tiny clusters |
| ranking_score | lower | combined selection score |

**Top 15 model runs**

| image_id | k | color_space | use_xy | silhouette_sample | davies_bouldin_sample | inertia_per_pixel | cluster_balance | ranking_score |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | lab | False | 0.7680300230392144 | 0.3889383730290883 | 3397.796426673979 | 0.9824740167108212 | 1.4515519988570056 |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | lab | True | 0.7585101716545529 | 0.397228702572854 | 3467.828345595299 | 0.9824740167108212 | 1.4714017305432043 |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | rgb | False | 0.7768994629167776 | 0.2886640635269787 | 766.5654205675322 | 0.648 | 1.5280869720455237 |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | rgb | True | 0.7543469492889678 | 0.3209496396153748 | 838.4400671829987 | 0.6494432628549981 | 1.5904329357596856 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | rgb | False | 0.663717964580371 | 0.450641220531663 | 2001.840447158684 | 0.9514781917009149 | 1.5956772318061554 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | lab | False | 0.6591137989567841 | 0.4640409341143357 | 1104.0615972266949 | 0.885970531710442 | 1.619722239096586 |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | rgb | False | 0.6615684530453798 | 0.4770289328753115 | 2524.9980681173874 | 0.9713716252944374 | 1.6275280083465953 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | rgb | True | 0.6525974237422562 | 0.4670227509144705 | 2090.7033874196254 | 0.9507023588656242 | 1.6282963713982146 |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | rgb | True | 0.6512798551101667 | 0.4917372826275049 | 2608.9041932243795 | 0.9731592310482408 | 1.6540050756757108 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | lab | True | 0.635919186911147 | 0.5002377159598438 | 1192.7235343260672 | 0.8852459016393442 | 1.6875558995444644 |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | lab | False | 0.6424904349545513 | 0.464546509915471 | 985.383811400412 | 0.7890724269377383 | 1.7223882020147208 |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | rgb | False | 0.6614702077130843 | 0.4464239423673495 | 2420.6202740715503 | 0.814959234314073 | 1.7492137133254575 |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 2 | rgb | False | 0.6463516030822222 | 0.4776364985860166 | 1262.2506603908505 | 0.7879869625950644 | 1.7574422896030486 |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 2 | lab | False | 0.7852136269873675 | 0.3137703042279379 | 2459.2277244384554 | 0.5518394648829431 | 1.7575181319128204 |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 2 | rgb | False | 0.6245320484158806 | 0.5304142507342385 | 4330.663151779813 | 0.9675491033304868 | 1.7757037836846037 |

## 9. Best Model Selection

The best model for each image is the run with the lowest `ranking_score`. This balances separation, compactness, and cluster stability.

| image_id | k | color_space | use_xy | silhouette_sample | davies_bouldin_sample | ranking_score |
| --- | --- | --- | --- | --- | --- | --- |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | lab | False | 0.7680300230392144 | 0.3889383730290883 | 1.4515519988570056 |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | rgb | False | 0.7768994629167776 | 0.2886640635269787 | 1.5280869720455237 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | rgb | False | 0.663717964580371 | 0.450641220531663 | 1.5956772318061554 |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | rgb | False | 0.6615684530453798 | 0.4770289328753115 | 1.6275280083465953 |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | lab | False | 0.6424904349545513 | 0.464546509915471 | 1.7223882020147208 |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | rgb | False | 0.6614702077130843 | 0.4464239423673495 | 1.7492137133254575 |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 2 | rgb | False | 0.6463516030822222 | 0.4776364985860166 | 1.7574422896030486 |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 2 | lab | False | 0.7852136269873675 | 0.3137703042279379 | 1.7575181319128204 |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 2 | rgb | False | 0.6245320484158806 | 0.5304142507342385 | 1.7757037836846037 |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 3 | lab | False | 0.875077260570786 | 0.2084735871637714 | 1.8488095325818388 |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 3 | lab | False | 0.8390200224812907 | 0.2337744664684616 | 1.8512513132609527 |
| Bontecou_Lake_Milky_Way_panorama_jpg | 3 | lab | False | 0.8359389243991842 | 0.2633130697616206 | 1.978389245044346 |
| Castle_Mountain_jpg | 2 | lab | False | 0.6381320390338326 | 0.6136415221891759 | 2.072480937059499 |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 2 | rgb | False | 0.5400444062908873 | 0.6752301069700718 | 2.08084331621532 |
| jpg | 2 | lab | False | 0.6729276680462352 | 0.5878205322606979 | 2.101153243700912 |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 2 | rgb | False | 0.7549214309460072 | 0.3563897037397857 | 2.105833867868676 |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 2 | rgb | False | 0.5537416068602463 | 0.6251626015634806 | 2.142069657872005 |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 2 | lab | False | 0.7767430361162948 | 0.2713149258660801 | 2.1538381192350964 |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 2 | lab | False | 0.6349555459172955 | 0.6678555841204598 | 2.273377819370455 |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 3 | lab | False | 0.8103970887812879 | 0.3781680639510036 | 2.280080992452206 |

## 10. Best Output Image Gallery

These figures show the best-ranked side-by-side output for each selected image.

| image_id | k | color_space | use_xy | ranking_score | Figure |
| --- | --- | --- | --- | --- | --- |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | lab | False | 1.4515519988570056 | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_lab_color_comparison.png" width="220"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | rgb | False | 1.5280869720455237 | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_rgb_color_comparison.png" width="220"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | rgb | False | 1.5956772318061554 | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_rgb_color_comparison.png" width="220"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | rgb | False | 1.6275280083465953 | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_rgb_color_comparison.png" width="220"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | lab | False | 1.7223882020147208 | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_lab_color_comparison.png" width="220"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | rgb | False | 1.7492137133254575 | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_rgb_color_comparison.png" width="220"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 2 | rgb | False | 1.7574422896030486 | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_rgb_color_comparison.png" width="220"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 2 | lab | False | 1.7575181319128204 | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_lab_color_comparison.png" width="220"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 2 | rgb | False | 1.7757037836846037 | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_rgb_color_comparison.png" width="220"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 3 | lab | False | 1.8488095325818388 | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_lab_color_comparison.png" width="220"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 3 | lab | False | 1.8512513132609527 | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_lab_color_comparison.png" width="220"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 3 | lab | False | 1.978389245044346 | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_lab_color_comparison.png" width="220"> |
| Castle_Mountain_jpg | 2 | lab | False | 2.072480937059499 | <img src="../reports/figures/Castle_Mountain_jpg_k2_lab_color_comparison.png" width="220"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 2 | rgb | False | 2.08084331621532 | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_rgb_color_comparison.png" width="220"> |
| jpg | 2 | lab | False | 2.101153243700912 | <img src="../reports/figures/jpg_k2_lab_color_comparison.png" width="220"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 2 | rgb | False | 2.105833867868676 | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_rgb_color_comparison.png" width="220"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 2 | rgb | False | 2.142069657872005 | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_rgb_color_comparison.png" width="220"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 2 | lab | False | 2.1538381192350964 | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_lab_color_comparison.png" width="220"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 2 | lab | False | 2.273377819370455 | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_lab_color_comparison.png" width="220"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 3 | lab | False | 2.280080992452206 | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_lab_color_comparison.png" width="220"> |

## 11. K-Grid Highlights

K-grid figures compare how segmentation changes as `K` increases.

<p><img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_color.png" width="260"><br>004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_color.png</p>\n<p><img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_xy.png" width="260"><br>004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_xy.png</p>\n<p><img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_color.png" width="260"><br>004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_color.png</p>\n<p><img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_xy.png" width="260"><br>004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_xy.png</p>\n<p><img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_color.png" width="260"><br>004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_color.png</p>\n<p><img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_xy.png" width="260"><br>004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_xy.png</p>\n<p><img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_hsv_color.png" width="260"><br>Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_hsv_color.png</p>\n<p><img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_hsv_xy.png" width="260"><br>Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_hsv_xy.png</p>\n<p><img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_lab_color.png" width="260"><br>Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_lab_color.png</p>\n<p><img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_lab_xy.png" width="260"><br>Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_lab_xy.png</p>\n<p><img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_rgb_color.png" width="260"><br>Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_rgb_color.png</p>\n<p><img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_rgb_xy.png" width="260"><br>Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_rgb_xy.png</p>\n<p><img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_hsv_color.png" width="260"><br>Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_hsv_color.png</p>\n<p><img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_hsv_xy.png" width="260"><br>Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_hsv_xy.png</p>\n<p><img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_lab_color.png" width="260"><br>Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_lab_color.png</p>\n<p><img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_lab_xy.png" width="260"><br>Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_lab_xy.png</p>\n<p><img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_rgb_color.png" width="260"><br>Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_rgb_color.png</p>\n<p><img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_rgb_xy.png" width="260"><br>Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_rgb_xy.png</p>\n<p><img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_hsv_color.png" width="260"><br>Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_hsv_color.png</p>\n<p><img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_hsv_xy.png" width="260"><br>Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_hsv_xy.png</p>\n<p><img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_lab_color.png" width="260"><br>Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_lab_color.png</p>\n<p><img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_lab_xy.png" width="260"><br>Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_lab_xy.png</p>\n<p><img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_rgb_color.png" width="260"><br>Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_rgb_color.png</p>\n<p><img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_rgb_xy.png" width="260"><br>Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_rgb_xy.png</p>\n

## 12. Output Image Gallery - All Trained Figures

This appendix lists every trained side-by-side comparison image generated by the retraining workflow.

| image_id | k | color_space | use_xy | Figure |
| --- | --- | --- | --- | --- |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 2 | hsv | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_hsv_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 3 | hsv | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k3_hsv_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 4 | hsv | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k4_hsv_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 5 | hsv | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k5_hsv_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 6 | hsv | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k6_hsv_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 7 | hsv | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k7_hsv_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 8 | hsv | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k8_hsv_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 9 | hsv | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k9_hsv_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 10 | hsv | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k10_hsv_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 2 | hsv | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_hsv_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 3 | hsv | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k3_hsv_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 4 | hsv | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k4_hsv_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 5 | hsv | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k5_hsv_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 6 | hsv | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k6_hsv_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 7 | hsv | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k7_hsv_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 8 | hsv | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k8_hsv_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 9 | hsv | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k9_hsv_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 10 | hsv | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k10_hsv_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 2 | lab | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_lab_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 3 | lab | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k3_lab_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 4 | lab | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k4_lab_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 5 | lab | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k5_lab_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 6 | lab | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k6_lab_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 7 | lab | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k7_lab_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 8 | lab | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k8_lab_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 9 | lab | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k9_lab_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 10 | lab | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k10_lab_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 2 | lab | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_lab_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 3 | lab | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k3_lab_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 4 | lab | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k4_lab_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 5 | lab | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k5_lab_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 6 | lab | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k6_lab_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 7 | lab | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k7_lab_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 8 | lab | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k8_lab_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 9 | lab | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k9_lab_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 10 | lab | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k10_lab_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 2 | rgb | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_rgb_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 3 | rgb | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k3_rgb_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 4 | rgb | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k4_rgb_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 5 | rgb | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k5_rgb_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 6 | rgb | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k6_rgb_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 7 | rgb | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k7_rgb_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 8 | rgb | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k8_rgb_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 9 | rgb | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k9_rgb_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 10 | rgb | False | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k10_rgb_color_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 2 | rgb | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_rgb_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 3 | rgb | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k3_rgb_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 4 | rgb | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k4_rgb_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 5 | rgb | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k5_rgb_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 6 | rgb | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k6_rgb_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 7 | rgb | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k7_rgb_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 8 | rgb | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k8_rgb_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 9 | rgb | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k9_rgb_xy_comparison.png" width="150"> |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 10 | rgb | True | <img src="../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | hsv | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_hsv_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 3 | hsv | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k3_hsv_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 4 | hsv | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k4_hsv_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 5 | hsv | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k5_hsv_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 6 | hsv | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k6_hsv_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 7 | hsv | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k7_hsv_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 8 | hsv | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k8_hsv_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 9 | hsv | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k9_hsv_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 10 | hsv | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k10_hsv_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | hsv | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_hsv_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 3 | hsv | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k3_hsv_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 4 | hsv | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k4_hsv_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 5 | hsv | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k5_hsv_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 6 | hsv | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k6_hsv_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 7 | hsv | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k7_hsv_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 8 | hsv | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k8_hsv_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 9 | hsv | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k9_hsv_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 10 | hsv | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k10_hsv_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | lab | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_lab_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 3 | lab | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k3_lab_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 4 | lab | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k4_lab_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 5 | lab | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k5_lab_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 6 | lab | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k6_lab_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 7 | lab | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k7_lab_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 8 | lab | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k8_lab_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 9 | lab | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k9_lab_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 10 | lab | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k10_lab_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | lab | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_lab_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 3 | lab | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k3_lab_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 4 | lab | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k4_lab_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 5 | lab | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k5_lab_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 6 | lab | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k6_lab_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 7 | lab | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k7_lab_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 8 | lab | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k8_lab_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 9 | lab | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k9_lab_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 10 | lab | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k10_lab_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | rgb | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_rgb_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 3 | rgb | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k3_rgb_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 4 | rgb | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k4_rgb_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 5 | rgb | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k5_rgb_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 6 | rgb | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k6_rgb_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 7 | rgb | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k7_rgb_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 8 | rgb | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k8_rgb_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 9 | rgb | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k9_rgb_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 10 | rgb | False | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k10_rgb_color_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | rgb | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_rgb_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 3 | rgb | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k3_rgb_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 4 | rgb | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k4_rgb_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 5 | rgb | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k5_rgb_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 6 | rgb | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k6_rgb_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 7 | rgb | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k7_rgb_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 8 | rgb | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k8_rgb_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 9 | rgb | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k9_rgb_xy_comparison.png" width="150"> |
| Algoma_Gabrielle_Rock_I0012362_jpg | 10 | rgb | True | <img src="../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | hsv | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_hsv_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 3 | hsv | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k3_hsv_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 4 | hsv | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k4_hsv_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 5 | hsv | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k5_hsv_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 6 | hsv | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k6_hsv_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 7 | hsv | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k7_hsv_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 8 | hsv | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k8_hsv_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 9 | hsv | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k9_hsv_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 10 | hsv | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k10_hsv_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | hsv | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_hsv_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 3 | hsv | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k3_hsv_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 4 | hsv | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k4_hsv_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 5 | hsv | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k5_hsv_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 6 | hsv | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k6_hsv_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 7 | hsv | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k7_hsv_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 8 | hsv | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k8_hsv_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 9 | hsv | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k9_hsv_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 10 | hsv | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k10_hsv_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | lab | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_lab_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 3 | lab | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k3_lab_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 4 | lab | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k4_lab_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 5 | lab | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k5_lab_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 6 | lab | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k6_lab_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 7 | lab | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k7_lab_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 8 | lab | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k8_lab_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 9 | lab | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k9_lab_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 10 | lab | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k10_lab_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | lab | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_lab_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 3 | lab | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k3_lab_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 4 | lab | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k4_lab_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 5 | lab | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k5_lab_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 6 | lab | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k6_lab_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 7 | lab | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k7_lab_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 8 | lab | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k8_lab_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 9 | lab | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k9_lab_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 10 | lab | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k10_lab_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | rgb | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_rgb_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 3 | rgb | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k3_rgb_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 4 | rgb | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k4_rgb_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 5 | rgb | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k5_rgb_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 6 | rgb | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k6_rgb_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 7 | rgb | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k7_rgb_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 8 | rgb | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k8_rgb_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 9 | rgb | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k9_rgb_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 10 | rgb | False | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k10_rgb_color_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | rgb | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_rgb_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 3 | rgb | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k3_rgb_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 4 | rgb | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k4_rgb_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 5 | rgb | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k5_rgb_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 6 | rgb | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k6_rgb_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 7 | rgb | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k7_rgb_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 8 | rgb | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k8_rgb_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 9 | rgb | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k9_rgb_xy_comparison.png" width="150"> |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 10 | rgb | True | <img src="../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 2 | hsv | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_hsv_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 3 | hsv | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k3_hsv_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 4 | hsv | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k4_hsv_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 5 | hsv | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k5_hsv_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 6 | hsv | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k6_hsv_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 7 | hsv | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k7_hsv_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 8 | hsv | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k8_hsv_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 9 | hsv | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k9_hsv_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 10 | hsv | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k10_hsv_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 2 | hsv | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_hsv_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 3 | hsv | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k3_hsv_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 4 | hsv | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k4_hsv_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 5 | hsv | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k5_hsv_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 6 | hsv | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k6_hsv_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 7 | hsv | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k7_hsv_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 8 | hsv | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k8_hsv_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 9 | hsv | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k9_hsv_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 10 | hsv | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k10_hsv_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 2 | lab | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_lab_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 3 | lab | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k3_lab_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 4 | lab | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k4_lab_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 5 | lab | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k5_lab_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 6 | lab | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k6_lab_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 7 | lab | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k7_lab_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 8 | lab | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k8_lab_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 9 | lab | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k9_lab_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 10 | lab | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k10_lab_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 2 | lab | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_lab_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 3 | lab | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k3_lab_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 4 | lab | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k4_lab_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 5 | lab | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k5_lab_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 6 | lab | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k6_lab_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 7 | lab | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k7_lab_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 8 | lab | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k8_lab_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 9 | lab | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k9_lab_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 10 | lab | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k10_lab_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 2 | rgb | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_rgb_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 3 | rgb | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k3_rgb_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 4 | rgb | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k4_rgb_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 5 | rgb | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k5_rgb_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 6 | rgb | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k6_rgb_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 7 | rgb | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k7_rgb_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 8 | rgb | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k8_rgb_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 9 | rgb | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k9_rgb_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 10 | rgb | False | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k10_rgb_color_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 2 | rgb | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_rgb_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 3 | rgb | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k3_rgb_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 4 | rgb | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k4_rgb_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 5 | rgb | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k5_rgb_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 6 | rgb | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k6_rgb_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 7 | rgb | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k7_rgb_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 8 | rgb | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k8_rgb_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 9 | rgb | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k9_rgb_xy_comparison.png" width="150"> |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 10 | rgb | True | <img src="../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 2 | hsv | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k2_hsv_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 3 | hsv | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_hsv_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 4 | hsv | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k4_hsv_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 5 | hsv | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k5_hsv_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 6 | hsv | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k6_hsv_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 7 | hsv | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k7_hsv_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 8 | hsv | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k8_hsv_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 9 | hsv | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k9_hsv_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 10 | hsv | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k10_hsv_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 2 | hsv | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k2_hsv_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 3 | hsv | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_hsv_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 4 | hsv | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k4_hsv_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 5 | hsv | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k5_hsv_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 6 | hsv | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k6_hsv_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 7 | hsv | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k7_hsv_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 8 | hsv | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k8_hsv_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 9 | hsv | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k9_hsv_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 10 | hsv | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k10_hsv_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 2 | lab | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k2_lab_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 3 | lab | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_lab_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 4 | lab | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k4_lab_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 5 | lab | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k5_lab_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 6 | lab | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k6_lab_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 7 | lab | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k7_lab_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 8 | lab | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k8_lab_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 9 | lab | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k9_lab_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 10 | lab | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k10_lab_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 2 | lab | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k2_lab_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 3 | lab | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_lab_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 4 | lab | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k4_lab_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 5 | lab | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k5_lab_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 6 | lab | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k6_lab_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 7 | lab | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k7_lab_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 8 | lab | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k8_lab_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 9 | lab | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k9_lab_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 10 | lab | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k10_lab_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 2 | rgb | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k2_rgb_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 3 | rgb | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_rgb_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 4 | rgb | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k4_rgb_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 5 | rgb | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k5_rgb_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 6 | rgb | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k6_rgb_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 7 | rgb | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k7_rgb_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 8 | rgb | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k8_rgb_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 9 | rgb | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k9_rgb_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 10 | rgb | False | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k10_rgb_color_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 2 | rgb | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k2_rgb_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 3 | rgb | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_rgb_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 4 | rgb | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k4_rgb_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 5 | rgb | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k5_rgb_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 6 | rgb | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k6_rgb_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 7 | rgb | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k7_rgb_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 8 | rgb | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k8_rgb_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 9 | rgb | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k9_rgb_xy_comparison.png" width="150"> |
| Bontecou_Lake_Milky_Way_panorama_jpg | 10 | rgb | True | <img src="../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k10_rgb_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | hsv | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_hsv_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 3 | hsv | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k3_hsv_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 4 | hsv | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k4_hsv_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 5 | hsv | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k5_hsv_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 6 | hsv | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k6_hsv_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 7 | hsv | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k7_hsv_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 8 | hsv | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k8_hsv_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 9 | hsv | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k9_hsv_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 10 | hsv | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k10_hsv_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | hsv | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_hsv_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 3 | hsv | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k3_hsv_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 4 | hsv | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k4_hsv_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 5 | hsv | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k5_hsv_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 6 | hsv | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k6_hsv_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 7 | hsv | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k7_hsv_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 8 | hsv | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k8_hsv_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 9 | hsv | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k9_hsv_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 10 | hsv | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k10_hsv_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | lab | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_lab_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 3 | lab | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k3_lab_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 4 | lab | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k4_lab_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 5 | lab | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k5_lab_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 6 | lab | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k6_lab_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 7 | lab | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k7_lab_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 8 | lab | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k8_lab_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 9 | lab | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k9_lab_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 10 | lab | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k10_lab_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | lab | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_lab_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 3 | lab | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k3_lab_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 4 | lab | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k4_lab_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 5 | lab | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k5_lab_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 6 | lab | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k6_lab_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 7 | lab | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k7_lab_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 8 | lab | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k8_lab_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 9 | lab | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k9_lab_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 10 | lab | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k10_lab_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | rgb | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_rgb_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 3 | rgb | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k3_rgb_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 4 | rgb | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k4_rgb_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 5 | rgb | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k5_rgb_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 6 | rgb | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k6_rgb_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 7 | rgb | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k7_rgb_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 8 | rgb | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k8_rgb_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 9 | rgb | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k9_rgb_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 10 | rgb | False | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k10_rgb_color_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | rgb | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_rgb_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 3 | rgb | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k3_rgb_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 4 | rgb | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k4_rgb_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 5 | rgb | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k5_rgb_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 6 | rgb | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k6_rgb_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 7 | rgb | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k7_rgb_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 8 | rgb | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k8_rgb_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 9 | rgb | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k9_rgb_xy_comparison.png" width="150"> |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 10 | rgb | True | <img src="../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 2 | hsv | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_hsv_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 3 | hsv | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k3_hsv_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 4 | hsv | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k4_hsv_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 5 | hsv | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k5_hsv_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 6 | hsv | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k6_hsv_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 7 | hsv | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k7_hsv_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 8 | hsv | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k8_hsv_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 9 | hsv | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k9_hsv_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 10 | hsv | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k10_hsv_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 2 | hsv | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_hsv_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 3 | hsv | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k3_hsv_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 4 | hsv | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k4_hsv_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 5 | hsv | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k5_hsv_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 6 | hsv | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k6_hsv_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 7 | hsv | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k7_hsv_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 8 | hsv | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k8_hsv_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 9 | hsv | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k9_hsv_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 10 | hsv | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k10_hsv_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 2 | lab | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_lab_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 3 | lab | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k3_lab_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 4 | lab | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k4_lab_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 5 | lab | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k5_lab_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 6 | lab | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k6_lab_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 7 | lab | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k7_lab_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 8 | lab | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k8_lab_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 9 | lab | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k9_lab_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 10 | lab | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k10_lab_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 2 | lab | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_lab_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 3 | lab | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k3_lab_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 4 | lab | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k4_lab_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 5 | lab | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k5_lab_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 6 | lab | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k6_lab_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 7 | lab | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k7_lab_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 8 | lab | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k8_lab_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 9 | lab | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k9_lab_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 10 | lab | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k10_lab_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 2 | rgb | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_rgb_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 3 | rgb | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k3_rgb_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 4 | rgb | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k4_rgb_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 5 | rgb | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k5_rgb_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 6 | rgb | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k6_rgb_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 7 | rgb | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k7_rgb_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 8 | rgb | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k8_rgb_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 9 | rgb | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k9_rgb_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 10 | rgb | False | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k10_rgb_color_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 2 | rgb | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_rgb_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 3 | rgb | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k3_rgb_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 4 | rgb | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k4_rgb_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 5 | rgb | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k5_rgb_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 6 | rgb | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k6_rgb_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 7 | rgb | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k7_rgb_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 8 | rgb | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k8_rgb_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 9 | rgb | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k9_rgb_xy_comparison.png" width="150"> |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 10 | rgb | True | <img src="../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k10_rgb_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 2 | hsv | False | <img src="../reports/figures/Castle_Mountain_jpg_k2_hsv_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 3 | hsv | False | <img src="../reports/figures/Castle_Mountain_jpg_k3_hsv_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 4 | hsv | False | <img src="../reports/figures/Castle_Mountain_jpg_k4_hsv_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 5 | hsv | False | <img src="../reports/figures/Castle_Mountain_jpg_k5_hsv_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 6 | hsv | False | <img src="../reports/figures/Castle_Mountain_jpg_k6_hsv_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 7 | hsv | False | <img src="../reports/figures/Castle_Mountain_jpg_k7_hsv_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 8 | hsv | False | <img src="../reports/figures/Castle_Mountain_jpg_k8_hsv_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 9 | hsv | False | <img src="../reports/figures/Castle_Mountain_jpg_k9_hsv_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 10 | hsv | False | <img src="../reports/figures/Castle_Mountain_jpg_k10_hsv_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 2 | hsv | True | <img src="../reports/figures/Castle_Mountain_jpg_k2_hsv_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 3 | hsv | True | <img src="../reports/figures/Castle_Mountain_jpg_k3_hsv_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 4 | hsv | True | <img src="../reports/figures/Castle_Mountain_jpg_k4_hsv_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 5 | hsv | True | <img src="../reports/figures/Castle_Mountain_jpg_k5_hsv_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 6 | hsv | True | <img src="../reports/figures/Castle_Mountain_jpg_k6_hsv_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 7 | hsv | True | <img src="../reports/figures/Castle_Mountain_jpg_k7_hsv_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 8 | hsv | True | <img src="../reports/figures/Castle_Mountain_jpg_k8_hsv_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 9 | hsv | True | <img src="../reports/figures/Castle_Mountain_jpg_k9_hsv_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 10 | hsv | True | <img src="../reports/figures/Castle_Mountain_jpg_k10_hsv_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 2 | lab | False | <img src="../reports/figures/Castle_Mountain_jpg_k2_lab_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 3 | lab | False | <img src="../reports/figures/Castle_Mountain_jpg_k3_lab_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 4 | lab | False | <img src="../reports/figures/Castle_Mountain_jpg_k4_lab_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 5 | lab | False | <img src="../reports/figures/Castle_Mountain_jpg_k5_lab_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 6 | lab | False | <img src="../reports/figures/Castle_Mountain_jpg_k6_lab_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 7 | lab | False | <img src="../reports/figures/Castle_Mountain_jpg_k7_lab_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 8 | lab | False | <img src="../reports/figures/Castle_Mountain_jpg_k8_lab_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 9 | lab | False | <img src="../reports/figures/Castle_Mountain_jpg_k9_lab_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 10 | lab | False | <img src="../reports/figures/Castle_Mountain_jpg_k10_lab_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 2 | lab | True | <img src="../reports/figures/Castle_Mountain_jpg_k2_lab_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 3 | lab | True | <img src="../reports/figures/Castle_Mountain_jpg_k3_lab_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 4 | lab | True | <img src="../reports/figures/Castle_Mountain_jpg_k4_lab_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 5 | lab | True | <img src="../reports/figures/Castle_Mountain_jpg_k5_lab_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 6 | lab | True | <img src="../reports/figures/Castle_Mountain_jpg_k6_lab_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 7 | lab | True | <img src="../reports/figures/Castle_Mountain_jpg_k7_lab_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 8 | lab | True | <img src="../reports/figures/Castle_Mountain_jpg_k8_lab_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 9 | lab | True | <img src="../reports/figures/Castle_Mountain_jpg_k9_lab_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 10 | lab | True | <img src="../reports/figures/Castle_Mountain_jpg_k10_lab_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 2 | rgb | False | <img src="../reports/figures/Castle_Mountain_jpg_k2_rgb_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 3 | rgb | False | <img src="../reports/figures/Castle_Mountain_jpg_k3_rgb_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 4 | rgb | False | <img src="../reports/figures/Castle_Mountain_jpg_k4_rgb_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 5 | rgb | False | <img src="../reports/figures/Castle_Mountain_jpg_k5_rgb_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 6 | rgb | False | <img src="../reports/figures/Castle_Mountain_jpg_k6_rgb_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 7 | rgb | False | <img src="../reports/figures/Castle_Mountain_jpg_k7_rgb_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 8 | rgb | False | <img src="../reports/figures/Castle_Mountain_jpg_k8_rgb_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 9 | rgb | False | <img src="../reports/figures/Castle_Mountain_jpg_k9_rgb_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 10 | rgb | False | <img src="../reports/figures/Castle_Mountain_jpg_k10_rgb_color_comparison.png" width="150"> |
| Castle_Mountain_jpg | 2 | rgb | True | <img src="../reports/figures/Castle_Mountain_jpg_k2_rgb_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 3 | rgb | True | <img src="../reports/figures/Castle_Mountain_jpg_k3_rgb_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 4 | rgb | True | <img src="../reports/figures/Castle_Mountain_jpg_k4_rgb_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 5 | rgb | True | <img src="../reports/figures/Castle_Mountain_jpg_k5_rgb_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 6 | rgb | True | <img src="../reports/figures/Castle_Mountain_jpg_k6_rgb_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 7 | rgb | True | <img src="../reports/figures/Castle_Mountain_jpg_k7_rgb_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 8 | rgb | True | <img src="../reports/figures/Castle_Mountain_jpg_k8_rgb_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 9 | rgb | True | <img src="../reports/figures/Castle_Mountain_jpg_k9_rgb_xy_comparison.png" width="150"> |
| Castle_Mountain_jpg | 10 | rgb | True | <img src="../reports/figures/Castle_Mountain_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | hsv | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_hsv_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 3 | hsv | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k3_hsv_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 4 | hsv | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k4_hsv_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 5 | hsv | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k5_hsv_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 6 | hsv | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k6_hsv_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 7 | hsv | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k7_hsv_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 8 | hsv | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k8_hsv_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 9 | hsv | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k9_hsv_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 10 | hsv | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k10_hsv_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | hsv | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_hsv_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 3 | hsv | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k3_hsv_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 4 | hsv | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k4_hsv_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 5 | hsv | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k5_hsv_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 6 | hsv | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k6_hsv_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 7 | hsv | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k7_hsv_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 8 | hsv | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k8_hsv_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 9 | hsv | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k9_hsv_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 10 | hsv | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k10_hsv_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | lab | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_lab_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 3 | lab | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k3_lab_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 4 | lab | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k4_lab_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 5 | lab | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k5_lab_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 6 | lab | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k6_lab_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 7 | lab | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k7_lab_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 8 | lab | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k8_lab_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 9 | lab | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k9_lab_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 10 | lab | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k10_lab_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | lab | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_lab_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 3 | lab | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k3_lab_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 4 | lab | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k4_lab_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 5 | lab | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k5_lab_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 6 | lab | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k6_lab_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 7 | lab | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k7_lab_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 8 | lab | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k8_lab_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 9 | lab | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k9_lab_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 10 | lab | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k10_lab_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | rgb | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_rgb_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 3 | rgb | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k3_rgb_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 4 | rgb | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k4_rgb_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 5 | rgb | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k5_rgb_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 6 | rgb | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k6_rgb_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 7 | rgb | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k7_rgb_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 8 | rgb | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k8_rgb_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 9 | rgb | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k9_rgb_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 10 | rgb | False | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k10_rgb_color_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | rgb | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_rgb_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 3 | rgb | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k3_rgb_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 4 | rgb | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k4_rgb_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 5 | rgb | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k5_rgb_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 6 | rgb | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k6_rgb_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 7 | rgb | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k7_rgb_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 8 | rgb | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k8_rgb_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 9 | rgb | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k9_rgb_xy_comparison.png" width="150"> |
| Catoctin_Mountain_and_farm_MD1_jpg | 10 | rgb | True | <img src="../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 2 | hsv | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_hsv_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 3 | hsv | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k3_hsv_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 4 | hsv | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k4_hsv_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 5 | hsv | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k5_hsv_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 6 | hsv | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k6_hsv_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 7 | hsv | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k7_hsv_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 8 | hsv | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k8_hsv_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 9 | hsv | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k9_hsv_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 10 | hsv | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k10_hsv_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 2 | hsv | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_hsv_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 3 | hsv | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k3_hsv_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 4 | hsv | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k4_hsv_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 5 | hsv | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k5_hsv_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 6 | hsv | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k6_hsv_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 7 | hsv | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k7_hsv_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 8 | hsv | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k8_hsv_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 9 | hsv | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k9_hsv_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 10 | hsv | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k10_hsv_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 2 | lab | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_lab_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 3 | lab | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k3_lab_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 4 | lab | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k4_lab_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 5 | lab | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k5_lab_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 6 | lab | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k6_lab_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 7 | lab | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k7_lab_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 8 | lab | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k8_lab_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 9 | lab | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k9_lab_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 10 | lab | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k10_lab_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 2 | lab | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_lab_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 3 | lab | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k3_lab_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 4 | lab | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k4_lab_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 5 | lab | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k5_lab_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 6 | lab | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k6_lab_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 7 | lab | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k7_lab_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 8 | lab | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k8_lab_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 9 | lab | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k9_lab_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 10 | lab | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k10_lab_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 2 | rgb | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_rgb_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 3 | rgb | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k3_rgb_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 4 | rgb | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k4_rgb_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 5 | rgb | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k5_rgb_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 6 | rgb | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k6_rgb_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 7 | rgb | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k7_rgb_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 8 | rgb | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k8_rgb_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 9 | rgb | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k9_rgb_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 10 | rgb | False | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k10_rgb_color_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 2 | rgb | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_rgb_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 3 | rgb | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k3_rgb_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 4 | rgb | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k4_rgb_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 5 | rgb | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k5_rgb_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 6 | rgb | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k6_rgb_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 7 | rgb | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k7_rgb_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 8 | rgb | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k8_rgb_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 9 | rgb | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k9_rgb_xy_comparison.png" width="150"> |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 10 | rgb | True | <img src="../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 2 | hsv | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_hsv_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 3 | hsv | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k3_hsv_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 4 | hsv | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k4_hsv_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 5 | hsv | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k5_hsv_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 6 | hsv | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k6_hsv_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 7 | hsv | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k7_hsv_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 8 | hsv | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k8_hsv_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 9 | hsv | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k9_hsv_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 10 | hsv | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k10_hsv_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 2 | hsv | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_hsv_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 3 | hsv | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k3_hsv_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 4 | hsv | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k4_hsv_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 5 | hsv | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k5_hsv_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 6 | hsv | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k6_hsv_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 7 | hsv | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k7_hsv_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 8 | hsv | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k8_hsv_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 9 | hsv | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k9_hsv_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 10 | hsv | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k10_hsv_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 2 | lab | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_lab_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 3 | lab | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k3_lab_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 4 | lab | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k4_lab_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 5 | lab | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k5_lab_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 6 | lab | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k6_lab_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 7 | lab | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k7_lab_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 8 | lab | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k8_lab_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 9 | lab | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k9_lab_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 10 | lab | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k10_lab_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 2 | lab | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_lab_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 3 | lab | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k3_lab_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 4 | lab | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k4_lab_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 5 | lab | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k5_lab_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 6 | lab | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k6_lab_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 7 | lab | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k7_lab_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 8 | lab | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k8_lab_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 9 | lab | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k9_lab_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 10 | lab | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k10_lab_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 2 | rgb | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_rgb_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 3 | rgb | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k3_rgb_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 4 | rgb | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k4_rgb_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 5 | rgb | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k5_rgb_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 6 | rgb | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k6_rgb_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 7 | rgb | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k7_rgb_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 8 | rgb | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k8_rgb_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 9 | rgb | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k9_rgb_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 10 | rgb | False | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k10_rgb_color_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 2 | rgb | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_rgb_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 3 | rgb | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k3_rgb_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 4 | rgb | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k4_rgb_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 5 | rgb | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k5_rgb_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 6 | rgb | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k6_rgb_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 7 | rgb | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k7_rgb_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 8 | rgb | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k8_rgb_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 9 | rgb | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k9_rgb_xy_comparison.png" width="150"> |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 10 | rgb | True | <img src="../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 2 | hsv | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_hsv_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 3 | hsv | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k3_hsv_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 4 | hsv | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k4_hsv_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 5 | hsv | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k5_hsv_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 6 | hsv | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k6_hsv_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 7 | hsv | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k7_hsv_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 8 | hsv | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k8_hsv_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 9 | hsv | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k9_hsv_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 10 | hsv | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k10_hsv_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 2 | hsv | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_hsv_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 3 | hsv | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k3_hsv_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 4 | hsv | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k4_hsv_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 5 | hsv | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k5_hsv_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 6 | hsv | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k6_hsv_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 7 | hsv | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k7_hsv_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 8 | hsv | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k8_hsv_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 9 | hsv | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k9_hsv_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 10 | hsv | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k10_hsv_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 2 | lab | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_lab_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 3 | lab | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k3_lab_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 4 | lab | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k4_lab_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 5 | lab | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k5_lab_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 6 | lab | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k6_lab_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 7 | lab | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k7_lab_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 8 | lab | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k8_lab_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 9 | lab | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k9_lab_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 10 | lab | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k10_lab_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 2 | lab | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_lab_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 3 | lab | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k3_lab_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 4 | lab | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k4_lab_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 5 | lab | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k5_lab_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 6 | lab | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k6_lab_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 7 | lab | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k7_lab_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 8 | lab | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k8_lab_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 9 | lab | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k9_lab_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 10 | lab | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k10_lab_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 2 | rgb | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_rgb_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 3 | rgb | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k3_rgb_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 4 | rgb | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k4_rgb_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 5 | rgb | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k5_rgb_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 6 | rgb | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k6_rgb_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 7 | rgb | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k7_rgb_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 8 | rgb | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k8_rgb_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 9 | rgb | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k9_rgb_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 10 | rgb | False | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k10_rgb_color_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 2 | rgb | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_rgb_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 3 | rgb | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k3_rgb_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 4 | rgb | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k4_rgb_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 5 | rgb | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k5_rgb_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 6 | rgb | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k6_rgb_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 7 | rgb | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k7_rgb_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 8 | rgb | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k8_rgb_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 9 | rgb | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k9_rgb_xy_comparison.png" width="150"> |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 10 | rgb | True | <img src="../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k10_rgb_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 2 | hsv | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k2_hsv_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 3 | hsv | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_hsv_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 4 | hsv | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k4_hsv_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 5 | hsv | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k5_hsv_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 6 | hsv | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k6_hsv_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 7 | hsv | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k7_hsv_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 8 | hsv | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k8_hsv_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 9 | hsv | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k9_hsv_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 10 | hsv | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k10_hsv_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 2 | hsv | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k2_hsv_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 3 | hsv | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_hsv_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 4 | hsv | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k4_hsv_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 5 | hsv | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k5_hsv_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 6 | hsv | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k6_hsv_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 7 | hsv | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k7_hsv_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 8 | hsv | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k8_hsv_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 9 | hsv | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k9_hsv_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 10 | hsv | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k10_hsv_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 2 | lab | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k2_lab_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 3 | lab | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_lab_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 4 | lab | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k4_lab_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 5 | lab | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k5_lab_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 6 | lab | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k6_lab_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 7 | lab | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k7_lab_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 8 | lab | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k8_lab_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 9 | lab | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k9_lab_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 10 | lab | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k10_lab_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 2 | lab | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k2_lab_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 3 | lab | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_lab_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 4 | lab | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k4_lab_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 5 | lab | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k5_lab_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 6 | lab | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k6_lab_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 7 | lab | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k7_lab_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 8 | lab | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k8_lab_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 9 | lab | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k9_lab_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 10 | lab | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k10_lab_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 2 | rgb | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k2_rgb_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 3 | rgb | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_rgb_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 4 | rgb | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k4_rgb_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 5 | rgb | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k5_rgb_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 6 | rgb | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k6_rgb_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 7 | rgb | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k7_rgb_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 8 | rgb | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k8_rgb_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 9 | rgb | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k9_rgb_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 10 | rgb | False | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k10_rgb_color_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 2 | rgb | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k2_rgb_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 3 | rgb | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_rgb_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 4 | rgb | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k4_rgb_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 5 | rgb | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k5_rgb_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 6 | rgb | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k6_rgb_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 7 | rgb | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k7_rgb_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 8 | rgb | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k8_rgb_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 9 | rgb | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k9_rgb_xy_comparison.png" width="150"> |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 10 | rgb | True | <img src="../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | hsv | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_hsv_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 3 | hsv | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k3_hsv_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 4 | hsv | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k4_hsv_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 5 | hsv | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k5_hsv_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 6 | hsv | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k6_hsv_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 7 | hsv | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k7_hsv_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 8 | hsv | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k8_hsv_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 9 | hsv | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k9_hsv_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 10 | hsv | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k10_hsv_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | hsv | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_hsv_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 3 | hsv | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k3_hsv_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 4 | hsv | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k4_hsv_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 5 | hsv | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k5_hsv_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 6 | hsv | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k6_hsv_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 7 | hsv | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k7_hsv_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 8 | hsv | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k8_hsv_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 9 | hsv | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k9_hsv_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 10 | hsv | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k10_hsv_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | lab | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_lab_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 3 | lab | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k3_lab_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 4 | lab | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k4_lab_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 5 | lab | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k5_lab_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 6 | lab | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k6_lab_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 7 | lab | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k7_lab_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 8 | lab | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k8_lab_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 9 | lab | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k9_lab_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 10 | lab | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k10_lab_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | lab | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_lab_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 3 | lab | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k3_lab_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 4 | lab | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k4_lab_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 5 | lab | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k5_lab_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 6 | lab | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k6_lab_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 7 | lab | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k7_lab_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 8 | lab | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k8_lab_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 9 | lab | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k9_lab_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 10 | lab | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k10_lab_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | rgb | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_rgb_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 3 | rgb | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k3_rgb_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 4 | rgb | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k4_rgb_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 5 | rgb | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k5_rgb_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 6 | rgb | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k6_rgb_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 7 | rgb | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k7_rgb_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 8 | rgb | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k8_rgb_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 9 | rgb | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k9_rgb_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 10 | rgb | False | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k10_rgb_color_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | rgb | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_rgb_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 3 | rgb | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k3_rgb_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 4 | rgb | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k4_rgb_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 5 | rgb | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k5_rgb_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 6 | rgb | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k6_rgb_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 7 | rgb | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k7_rgb_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 8 | rgb | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k8_rgb_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 9 | rgb | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k9_rgb_xy_comparison.png" width="150"> |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 10 | rgb | True | <img src="../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k10_rgb_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 2 | hsv | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k2_hsv_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 3 | hsv | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_hsv_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 4 | hsv | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k4_hsv_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 5 | hsv | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k5_hsv_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 6 | hsv | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k6_hsv_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 7 | hsv | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k7_hsv_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 8 | hsv | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k8_hsv_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 9 | hsv | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k9_hsv_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 10 | hsv | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k10_hsv_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 2 | hsv | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k2_hsv_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 3 | hsv | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_hsv_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 4 | hsv | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k4_hsv_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 5 | hsv | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k5_hsv_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 6 | hsv | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k6_hsv_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 7 | hsv | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k7_hsv_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 8 | hsv | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k8_hsv_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 9 | hsv | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k9_hsv_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 10 | hsv | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k10_hsv_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 2 | lab | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k2_lab_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 3 | lab | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_lab_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 4 | lab | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k4_lab_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 5 | lab | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k5_lab_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 6 | lab | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k6_lab_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 7 | lab | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k7_lab_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 8 | lab | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k8_lab_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 9 | lab | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k9_lab_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 10 | lab | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k10_lab_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 2 | lab | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k2_lab_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 3 | lab | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_lab_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 4 | lab | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k4_lab_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 5 | lab | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k5_lab_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 6 | lab | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k6_lab_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 7 | lab | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k7_lab_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 8 | lab | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k8_lab_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 9 | lab | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k9_lab_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 10 | lab | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k10_lab_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 2 | rgb | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k2_rgb_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 3 | rgb | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_rgb_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 4 | rgb | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k4_rgb_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 5 | rgb | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k5_rgb_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 6 | rgb | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k6_rgb_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 7 | rgb | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k7_rgb_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 8 | rgb | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k8_rgb_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 9 | rgb | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k9_rgb_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 10 | rgb | False | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k10_rgb_color_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 2 | rgb | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k2_rgb_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 3 | rgb | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_rgb_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 4 | rgb | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k4_rgb_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 5 | rgb | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k5_rgb_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 6 | rgb | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k6_rgb_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 7 | rgb | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k7_rgb_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 8 | rgb | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k8_rgb_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 9 | rgb | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k9_rgb_xy_comparison.png" width="150"> |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 10 | rgb | True | <img src="../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 2 | hsv | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_hsv_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 3 | hsv | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k3_hsv_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 4 | hsv | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k4_hsv_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 5 | hsv | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k5_hsv_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 6 | hsv | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k6_hsv_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 7 | hsv | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k7_hsv_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 8 | hsv | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k8_hsv_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 9 | hsv | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k9_hsv_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 10 | hsv | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k10_hsv_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 2 | hsv | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_hsv_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 3 | hsv | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k3_hsv_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 4 | hsv | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k4_hsv_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 5 | hsv | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k5_hsv_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 6 | hsv | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k6_hsv_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 7 | hsv | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k7_hsv_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 8 | hsv | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k8_hsv_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 9 | hsv | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k9_hsv_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 10 | hsv | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k10_hsv_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 2 | lab | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_lab_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 3 | lab | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k3_lab_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 4 | lab | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k4_lab_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 5 | lab | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k5_lab_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 6 | lab | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k6_lab_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 7 | lab | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k7_lab_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 8 | lab | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k8_lab_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 9 | lab | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k9_lab_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 10 | lab | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k10_lab_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 2 | lab | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_lab_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 3 | lab | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k3_lab_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 4 | lab | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k4_lab_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 5 | lab | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k5_lab_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 6 | lab | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k6_lab_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 7 | lab | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k7_lab_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 8 | lab | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k8_lab_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 9 | lab | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k9_lab_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 10 | lab | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k10_lab_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 2 | rgb | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_rgb_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 3 | rgb | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k3_rgb_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 4 | rgb | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k4_rgb_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 5 | rgb | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k5_rgb_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 6 | rgb | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k6_rgb_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 7 | rgb | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k7_rgb_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 8 | rgb | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k8_rgb_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 9 | rgb | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k9_rgb_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 10 | rgb | False | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k10_rgb_color_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 2 | rgb | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_rgb_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 3 | rgb | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k3_rgb_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 4 | rgb | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k4_rgb_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 5 | rgb | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k5_rgb_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 6 | rgb | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k6_rgb_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 7 | rgb | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k7_rgb_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 8 | rgb | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k8_rgb_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 9 | rgb | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k9_rgb_xy_comparison.png" width="150"> |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 10 | rgb | True | <img src="../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | hsv | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_hsv_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 3 | hsv | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k3_hsv_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 4 | hsv | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k4_hsv_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 5 | hsv | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k5_hsv_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 6 | hsv | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k6_hsv_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 7 | hsv | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k7_hsv_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 8 | hsv | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k8_hsv_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 9 | hsv | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k9_hsv_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 10 | hsv | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k10_hsv_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | hsv | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_hsv_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 3 | hsv | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k3_hsv_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 4 | hsv | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k4_hsv_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 5 | hsv | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k5_hsv_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 6 | hsv | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k6_hsv_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 7 | hsv | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k7_hsv_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 8 | hsv | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k8_hsv_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 9 | hsv | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k9_hsv_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 10 | hsv | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k10_hsv_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | lab | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_lab_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 3 | lab | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k3_lab_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 4 | lab | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k4_lab_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 5 | lab | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k5_lab_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 6 | lab | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k6_lab_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 7 | lab | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k7_lab_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 8 | lab | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k8_lab_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 9 | lab | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k9_lab_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 10 | lab | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k10_lab_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | lab | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_lab_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 3 | lab | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k3_lab_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 4 | lab | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k4_lab_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 5 | lab | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k5_lab_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 6 | lab | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k6_lab_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 7 | lab | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k7_lab_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 8 | lab | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k8_lab_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 9 | lab | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k9_lab_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 10 | lab | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k10_lab_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | rgb | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_rgb_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 3 | rgb | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k3_rgb_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 4 | rgb | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k4_rgb_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 5 | rgb | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k5_rgb_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 6 | rgb | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k6_rgb_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 7 | rgb | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k7_rgb_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 8 | rgb | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k8_rgb_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 9 | rgb | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k9_rgb_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 10 | rgb | False | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k10_rgb_color_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | rgb | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_rgb_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 3 | rgb | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k3_rgb_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 4 | rgb | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k4_rgb_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 5 | rgb | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k5_rgb_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 6 | rgb | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k6_rgb_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 7 | rgb | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k7_rgb_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 8 | rgb | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k8_rgb_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 9 | rgb | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k9_rgb_xy_comparison.png" width="150"> |
| Sunset_by_Caspar_David_Friedrich_jpg | 10 | rgb | True | <img src="../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k10_rgb_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 2 | hsv | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k2_hsv_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 3 | hsv | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_hsv_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 4 | hsv | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k4_hsv_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 5 | hsv | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k5_hsv_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 6 | hsv | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k6_hsv_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 7 | hsv | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k7_hsv_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 8 | hsv | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k8_hsv_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 9 | hsv | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k9_hsv_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 10 | hsv | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k10_hsv_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 2 | hsv | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k2_hsv_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 3 | hsv | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_hsv_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 4 | hsv | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k4_hsv_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 5 | hsv | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k5_hsv_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 6 | hsv | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k6_hsv_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 7 | hsv | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k7_hsv_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 8 | hsv | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k8_hsv_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 9 | hsv | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k9_hsv_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 10 | hsv | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k10_hsv_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 2 | lab | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k2_lab_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 3 | lab | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_lab_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 4 | lab | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k4_lab_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 5 | lab | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k5_lab_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 6 | lab | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k6_lab_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 7 | lab | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k7_lab_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 8 | lab | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k8_lab_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 9 | lab | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k9_lab_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 10 | lab | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k10_lab_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 2 | lab | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k2_lab_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 3 | lab | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_lab_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 4 | lab | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k4_lab_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 5 | lab | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k5_lab_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 6 | lab | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k6_lab_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 7 | lab | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k7_lab_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 8 | lab | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k8_lab_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 9 | lab | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k9_lab_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 10 | lab | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k10_lab_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 2 | rgb | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k2_rgb_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 3 | rgb | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_rgb_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 4 | rgb | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k4_rgb_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 5 | rgb | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k5_rgb_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 6 | rgb | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k6_rgb_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 7 | rgb | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k7_rgb_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 8 | rgb | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k8_rgb_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 9 | rgb | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k9_rgb_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 10 | rgb | False | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k10_rgb_color_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 2 | rgb | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k2_rgb_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 3 | rgb | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_rgb_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 4 | rgb | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k4_rgb_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 5 | rgb | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k5_rgb_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 6 | rgb | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k6_rgb_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 7 | rgb | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k7_rgb_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 8 | rgb | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k8_rgb_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 9 | rgb | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k9_rgb_xy_comparison.png" width="150"> |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 10 | rgb | True | <img src="../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k10_rgb_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 2 | hsv | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_hsv_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 3 | hsv | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k3_hsv_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 4 | hsv | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k4_hsv_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 5 | hsv | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k5_hsv_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 6 | hsv | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k6_hsv_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 7 | hsv | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k7_hsv_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 8 | hsv | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k8_hsv_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 9 | hsv | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k9_hsv_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 10 | hsv | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k10_hsv_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 2 | hsv | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_hsv_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 3 | hsv | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k3_hsv_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 4 | hsv | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k4_hsv_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 5 | hsv | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k5_hsv_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 6 | hsv | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k6_hsv_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 7 | hsv | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k7_hsv_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 8 | hsv | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k8_hsv_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 9 | hsv | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k9_hsv_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 10 | hsv | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k10_hsv_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 2 | lab | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_lab_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 3 | lab | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k3_lab_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 4 | lab | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k4_lab_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 5 | lab | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k5_lab_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 6 | lab | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k6_lab_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 7 | lab | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k7_lab_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 8 | lab | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k8_lab_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 9 | lab | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k9_lab_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 10 | lab | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k10_lab_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 2 | lab | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_lab_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 3 | lab | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k3_lab_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 4 | lab | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k4_lab_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 5 | lab | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k5_lab_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 6 | lab | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k6_lab_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 7 | lab | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k7_lab_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 8 | lab | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k8_lab_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 9 | lab | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k9_lab_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 10 | lab | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k10_lab_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 2 | rgb | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_rgb_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 3 | rgb | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k3_rgb_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 4 | rgb | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k4_rgb_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 5 | rgb | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k5_rgb_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 6 | rgb | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k6_rgb_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 7 | rgb | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k7_rgb_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 8 | rgb | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k8_rgb_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 9 | rgb | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k9_rgb_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 10 | rgb | False | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k10_rgb_color_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 2 | rgb | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_rgb_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 3 | rgb | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k3_rgb_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 4 | rgb | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k4_rgb_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 5 | rgb | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k5_rgb_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 6 | rgb | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k6_rgb_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 7 | rgb | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k7_rgb_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 8 | rgb | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k8_rgb_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 9 | rgb | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k9_rgb_xy_comparison.png" width="150"> |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 10 | rgb | True | <img src="../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k10_rgb_xy_comparison.png" width="150"> |
| jpg | 2 | hsv | False | <img src="../reports/figures/jpg_k2_hsv_color_comparison.png" width="150"> |
| jpg | 3 | hsv | False | <img src="../reports/figures/jpg_k3_hsv_color_comparison.png" width="150"> |
| jpg | 4 | hsv | False | <img src="../reports/figures/jpg_k4_hsv_color_comparison.png" width="150"> |
| jpg | 5 | hsv | False | <img src="../reports/figures/jpg_k5_hsv_color_comparison.png" width="150"> |
| jpg | 6 | hsv | False | <img src="../reports/figures/jpg_k6_hsv_color_comparison.png" width="150"> |
| jpg | 7 | hsv | False | <img src="../reports/figures/jpg_k7_hsv_color_comparison.png" width="150"> |
| jpg | 8 | hsv | False | <img src="../reports/figures/jpg_k8_hsv_color_comparison.png" width="150"> |
| jpg | 9 | hsv | False | <img src="../reports/figures/jpg_k9_hsv_color_comparison.png" width="150"> |
| jpg | 10 | hsv | False | <img src="../reports/figures/jpg_k10_hsv_color_comparison.png" width="150"> |
| jpg | 2 | hsv | True | <img src="../reports/figures/jpg_k2_hsv_xy_comparison.png" width="150"> |
| jpg | 3 | hsv | True | <img src="../reports/figures/jpg_k3_hsv_xy_comparison.png" width="150"> |
| jpg | 4 | hsv | True | <img src="../reports/figures/jpg_k4_hsv_xy_comparison.png" width="150"> |
| jpg | 5 | hsv | True | <img src="../reports/figures/jpg_k5_hsv_xy_comparison.png" width="150"> |
| jpg | 6 | hsv | True | <img src="../reports/figures/jpg_k6_hsv_xy_comparison.png" width="150"> |
| jpg | 7 | hsv | True | <img src="../reports/figures/jpg_k7_hsv_xy_comparison.png" width="150"> |
| jpg | 8 | hsv | True | <img src="../reports/figures/jpg_k8_hsv_xy_comparison.png" width="150"> |
| jpg | 9 | hsv | True | <img src="../reports/figures/jpg_k9_hsv_xy_comparison.png" width="150"> |
| jpg | 10 | hsv | True | <img src="../reports/figures/jpg_k10_hsv_xy_comparison.png" width="150"> |
| jpg | 2 | lab | False | <img src="../reports/figures/jpg_k2_lab_color_comparison.png" width="150"> |
| jpg | 3 | lab | False | <img src="../reports/figures/jpg_k3_lab_color_comparison.png" width="150"> |
| jpg | 4 | lab | False | <img src="../reports/figures/jpg_k4_lab_color_comparison.png" width="150"> |
| jpg | 5 | lab | False | <img src="../reports/figures/jpg_k5_lab_color_comparison.png" width="150"> |
| jpg | 6 | lab | False | <img src="../reports/figures/jpg_k6_lab_color_comparison.png" width="150"> |
| jpg | 7 | lab | False | <img src="../reports/figures/jpg_k7_lab_color_comparison.png" width="150"> |
| jpg | 8 | lab | False | <img src="../reports/figures/jpg_k8_lab_color_comparison.png" width="150"> |
| jpg | 9 | lab | False | <img src="../reports/figures/jpg_k9_lab_color_comparison.png" width="150"> |
| jpg | 10 | lab | False | <img src="../reports/figures/jpg_k10_lab_color_comparison.png" width="150"> |
| jpg | 2 | lab | True | <img src="../reports/figures/jpg_k2_lab_xy_comparison.png" width="150"> |
| jpg | 3 | lab | True | <img src="../reports/figures/jpg_k3_lab_xy_comparison.png" width="150"> |
| jpg | 4 | lab | True | <img src="../reports/figures/jpg_k4_lab_xy_comparison.png" width="150"> |
| jpg | 5 | lab | True | <img src="../reports/figures/jpg_k5_lab_xy_comparison.png" width="150"> |
| jpg | 6 | lab | True | <img src="../reports/figures/jpg_k6_lab_xy_comparison.png" width="150"> |
| jpg | 7 | lab | True | <img src="../reports/figures/jpg_k7_lab_xy_comparison.png" width="150"> |
| jpg | 8 | lab | True | <img src="../reports/figures/jpg_k8_lab_xy_comparison.png" width="150"> |
| jpg | 9 | lab | True | <img src="../reports/figures/jpg_k9_lab_xy_comparison.png" width="150"> |
| jpg | 10 | lab | True | <img src="../reports/figures/jpg_k10_lab_xy_comparison.png" width="150"> |
| jpg | 2 | rgb | False | <img src="../reports/figures/jpg_k2_rgb_color_comparison.png" width="150"> |
| jpg | 3 | rgb | False | <img src="../reports/figures/jpg_k3_rgb_color_comparison.png" width="150"> |
| jpg | 4 | rgb | False | <img src="../reports/figures/jpg_k4_rgb_color_comparison.png" width="150"> |
| jpg | 5 | rgb | False | <img src="../reports/figures/jpg_k5_rgb_color_comparison.png" width="150"> |
| jpg | 6 | rgb | False | <img src="../reports/figures/jpg_k6_rgb_color_comparison.png" width="150"> |
| jpg | 7 | rgb | False | <img src="../reports/figures/jpg_k7_rgb_color_comparison.png" width="150"> |
| jpg | 8 | rgb | False | <img src="../reports/figures/jpg_k8_rgb_color_comparison.png" width="150"> |
| jpg | 9 | rgb | False | <img src="../reports/figures/jpg_k9_rgb_color_comparison.png" width="150"> |
| jpg | 10 | rgb | False | <img src="../reports/figures/jpg_k10_rgb_color_comparison.png" width="150"> |
| jpg | 2 | rgb | True | <img src="../reports/figures/jpg_k2_rgb_xy_comparison.png" width="150"> |
| jpg | 3 | rgb | True | <img src="../reports/figures/jpg_k3_rgb_xy_comparison.png" width="150"> |
| jpg | 4 | rgb | True | <img src="../reports/figures/jpg_k4_rgb_xy_comparison.png" width="150"> |
| jpg | 5 | rgb | True | <img src="../reports/figures/jpg_k5_rgb_xy_comparison.png" width="150"> |
| jpg | 6 | rgb | True | <img src="../reports/figures/jpg_k6_rgb_xy_comparison.png" width="150"> |
| jpg | 7 | rgb | True | <img src="../reports/figures/jpg_k7_rgb_xy_comparison.png" width="150"> |
| jpg | 8 | rgb | True | <img src="../reports/figures/jpg_k8_rgb_xy_comparison.png" width="150"> |
| jpg | 9 | rgb | True | <img src="../reports/figures/jpg_k9_rgb_xy_comparison.png" width="150"> |
| jpg | 10 | rgb | True | <img src="../reports/figures/jpg_k10_rgb_xy_comparison.png" width="150"> |

## 13. Final Review

The review script checks data balance, expected model-run count, metric completeness, artifact existence, notebook structure, and assignment alignment.

| Check | Result |
|---|---|
| clean images | 20 |
| model runs | 1080 |
| expected model runs | 1080 |
| assignment alignment | K-Means image segmentation with landscape images |

The final notebook has no code cells. The project remains aligned with the original K-Means Image Segmentation exercise: load image, convert color space, resize, flatten pixels, cluster by K-Means, reconstruct segmented image, and visualize original versus segmented output.